In [ ]:
#SUP FIGURE 1C -  expression of key marker genes for : CD8 cluster

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import warnings

warnings.filterwarnings('ignore')

# Load the single-cell data
adata_sc_new = sc.read_h5ad("/Users/yashkulkarni/cellxgene_data/processed_deconvolution_sc_spleen_240713_fixed_full_data.h5ad")

print("Setting up CD8 T cell heatmap (INFECTED ONLY)...\n")

# CD8 cell types (excluding gamma-delta and NKT)
cd8_types_current = [
    'CD8-Tcell_naive',
    'CD8-Tcell_early-active',
    'CD8-Tcell_proliferating',
    'CD8-Tcell_late-active',
    'CD8-Tcell_effector'

]

# Gene list from reference (removing duplicates)
cd8_genes = [
    'Hdac9', 'Myb', 'Sox4', 'Itgae', 'Slc6a19',
    'Sell', 'Tcf7', 'Ccr7', 'Lef1', 'Il27ra', 'Ikzf2',  'Ifngas1', 'Bcl2', 'Eomes', 'Ifng',
    'Gzmb', 'Gzmk', 'Gzma', 'Cx3cr1', 'Prf1', 'Ccr5', 'Tbx21', 'Il18r1', 
    'Il18rap', 'Cd5', 'Txn1', 'Stmn1', 'Mki67', 'Cxcr5', 'Klrg1', 'Cd44', 'Tcf7', 'Cxcr3', 'Il12', 'Cd28', 'Icos'
]

# Remove duplicates while preserving order
cd8_genes_unique = list(dict.fromkeys(cd8_genes))

print(f"CD8 cell types: {len(cd8_types_current)}")
print(f"Genes: {len(cd8_genes_unique)}")

# FILTER FOR INFECTED CELLS ONLY (3wk timepoint)
print("\nFiltering for infected cells (3wk timepoint)...")
adata_infected = adata_sc_new[adata_sc_new.obs['timepoint'] == '3wk'].copy()
print(f"Total infected cells: {adata_infected.shape[0]}")

# Filter for CD8 cells - INFECTED ONLY
adata_cd8_infected = adata_infected[adata_infected.obs['celltypes_redo'].isin(cd8_types_current)].copy()
print(f"Infected CD8 cells: {adata_cd8_infected.shape[0]}")

print("\nCells per population (infected only):")
for ct in cd8_types_current:
    n_cells = (adata_cd8_infected.obs['celltypes_redo'] == ct).sum()
    print(f"  {ct}: {n_cells}")

# Check which genes are present
cd8_genes_present = [g for g in cd8_genes_unique if g in adata_cd8_infected.var_names]
cd8_genes_missing = [g for g in cd8_genes_unique if g not in adata_cd8_infected.var_names]

print(f"\nGenes present: {len(cd8_genes_present)}/{len(cd8_genes_unique)}")
if cd8_genes_missing:
    print(f"Missing genes ({len(cd8_genes_missing)}): {cd8_genes_missing}")

# Subset to genes
adata_cd8_subset = adata_cd8_infected[:, cd8_genes_present].copy()

# Calculate mean expression
print("\nCalculating mean expression...")
expr_matrix_cd8 = []
cell_type_labels_cd8 = []

for cell_type in cd8_types_current:
    cells_mask = adata_cd8_subset.obs['celltypes_redo'] == cell_type
    cells = adata_cd8_subset[cells_mask]
    
    if cells.shape[0] > 0:
        if hasattr(cells.X, 'toarray'):
            X = cells.X.toarray()
        else:
            X = cells.X
        
        mean_expr = X.mean(axis=0)
        expr_matrix_cd8.append(mean_expr)
        cell_type_labels_cd8.append(cell_type)

# Create dataframe
expr_df_cd8 = pd.DataFrame(
    expr_matrix_cd8,
    index=cell_type_labels_cd8,
    columns=cd8_genes_present
)

# Z-score normalize
expr_df_cd8_norm = expr_df_cd8.apply(zscore, axis=0)
expr_df_cd8_norm = expr_df_cd8_norm.fillna(0)

print(f"\nExpression matrix: {expr_df_cd8_norm.shape}")
print(f"Cell types: {expr_df_cd8_norm.index.tolist()}")
print("\n✅ CD8 data ready!")

# Now create the heatmap
print("\nCreating CD8 T cell heatmap (ordered by peak expression)...\n")

# For each gene, find which cell type has highest expression
gene_to_peak_celltype = {}
for gene in expr_df_cd8_norm.columns:
    peak_celltype = expr_df_cd8_norm[gene].idxmax()
    gene_to_peak_celltype[gene] = peak_celltype

# Group genes by their peak cell type (in order of cell type progression)
genes_by_celltype = {ct: [] for ct in cd8_types_current}
for gene, peak_ct in gene_to_peak_celltype.items():
    genes_by_celltype[peak_ct].append(gene)

# Concatenate genes in order of cell types
ordered_genes = []
for ct in cd8_types_current:
    ordered_genes.extend(genes_by_celltype[ct])
    print(f"{ct}: {len(genes_by_celltype[ct])} genes")

# Reorder columns by peak expression
expr_df_cd8_ordered = expr_df_cd8_norm[ordered_genes]

# Calculate actual z-score range (don't cut it off prematurely)
vmin = expr_df_cd8_ordered.min().min()
vmax = expr_df_cd8_ordered.max().max()

print(f"Z-score range: {vmin:.2f} to {vmax:.2f}")

# Create figure (NO BRACKETS)
fig, ax = plt.subplots(figsize=(14, 5))

sns.heatmap(
    expr_df_cd8_ordered,
    cmap='RdBu_r',
    vmin=vmin,  # Use actual data range
    vmax=vmax,  # Use actual data range
    cbar_kws={
        'label': 'Z-score',
        'shrink': 0.7,
        'pad': 0.01
    },
    linewidths=0,
    linecolor='white',
    square=False,
    ax=ax,
    yticklabels=True,
    xticklabels=True
)

ax.set_title('Selected Genes Across CD8 T Cell Types (Infected)', 
             fontsize=15, fontweight='bold', pad=10)

plt.xticks(rotation=90, ha='center', fontsize=11)
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()
plt.savefig('cd8_gene_expression_heatmap_infected_ordered.png', dpi=300, bbox_inches='tight')
plt.savefig('cd8_gene_expression_heatmap_infected_ordered.pdf', bbox_inches='tight')
print("\n📊 Saved: cd8_gene_expression_heatmap_infected_ordered.png/pdf")
plt.show()

print(f"\n✅ CD8 Heatmap (infected only, ordered by peak expression) complete!")

In [ ]:
#SUP FIGURE 1C COARSE CELLTYPES

import scanpy as sc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import OrderedDict
from scipy.stats import zscore

warnings.filterwarnings('ignore')

adata_sc_new = sc.read_h5ad("/Users/yashkulkarni/cellxgene_data/processed_deconvolution_sc_spleen_240713_fixed_full_data.h5ad")
adata_sc_new_inf = adata_sc_new[adata_sc_new.obs['timepoint'] == '3wk'].copy()

In [ ]:
#SUP FIGURE 1C COARSE CELLTYPES

# CORRECTED gene list from reference (left to right)
coarse_genes = [
    'Klrb1c', 'Klre1', 'Cd4', 'Cd8a', 'Cd8b1', 'Trbc2', 'Cd3d', 'Cd3e',
    'Themis', 'Cd79b', 'Cd19', 'Cd79a', 'Gclm', 'Prdx2', 'Cdh5', 'Stab2', 'Col1a1', 'Col1a2',
    'Cpa3', 'Cyp11a1', 'Csf3r', 'Csf1r', 'Cst3','Camp', 'Ngp', 'S100a8', 'S100a9'
]

# Coarse cell types (from image, top to bottom)
coarse_cell_types = [
    'NK',
    'CD4-Tcell',
    'CD8-Tcell',
    'Bcell',
    'Hematopoietic',
    'Endothelial',
    'Fibroblast',
    'Mast-Cell',
    'Myeloid',
    'Neutrophil'
]

print(f"Coarse cell types: {len(coarse_cell_types)}")
print(f"Genes: {len(coarse_genes)}")

# Filter for coarse cell types
adata_coarse = adata_sc_new[adata_sc_new.obs['coarse_redo'].isin(coarse_cell_types)].copy()
print(f"\nCoarse cells: {adata_coarse.shape[0]}")

# Check which genes are present
coarse_genes_present = [g for g in coarse_genes if g in adata_coarse.var_names]
coarse_genes_missing = [g for g in coarse_genes if g not in adata_coarse.var_names]

print(f"\nGenes present: {len(coarse_genes_present)}/{len(coarse_genes)}")
if coarse_genes_missing:
    print(f"Missing genes: {coarse_genes_missing}")

# Subset to genes
adata_coarse_subset = adata_coarse[:, coarse_genes_present].copy()

# Calculate mean expression
print("\nCalculating mean expression...")
expr_matrix_coarse = []
cell_type_labels_coarse = []

for cell_type in coarse_cell_types:
    cells_mask = adata_coarse_subset.obs['coarse_redo'] == cell_type
    cells = adata_coarse_subset[cells_mask]
    
    if cells.shape[0] > 0:
        if hasattr(cells.X, 'toarray'):
            X = cells.X.toarray()
        else:
            X = cells.X
        
        mean_expr = X.mean(axis=0)
        expr_matrix_coarse.append(mean_expr)
        cell_type_labels_coarse.append(cell_type)

# Create dataframe
expr_df_coarse = pd.DataFrame(
    expr_matrix_coarse,
    index=cell_type_labels_coarse,
    columns=coarse_genes_present
)

# Z-score normalize
expr_df_coarse_norm = expr_df_coarse.apply(zscore, axis=0)

print(f"\nExpression matrix: {expr_df_coarse_norm.shape}")
print(f"Cell types: {expr_df_coarse_norm.index.tolist()}")
print("\n✅ Coarse cluster data ready with corrected genes!")

In [ ]:
#SUP FIGURE 1C COARSE CELLTYPES
# Reorder rows and columns
expr_df_coarse_ordered = expr_df_coarse_norm.reindex(index=cell_type_labels_coarse, columns=coarse_genes_present)

# Calculate actual z-score range
vmin = expr_df_coarse_ordered.min().min()
vmax = expr_df_coarse_ordered.max().max()

print(f"Z-score range: {vmin:.2f} to {vmax:.2f}")

# Create figure (same dimensions as previous plots)
fig, ax = plt.subplots(figsize=(12, 5))

# Plot heatmap
sns.heatmap(
    expr_df_coarse_ordered,
    cmap='RdBu_r',
    vmin=vmin,  # Use actual data range
    vmax=vmax,  # Use actual data range
    cbar_kws={
        'label': 'Z-score',
        'shrink': 0.7,
        'pad': 0.01
    },
    linewidths=0.001,
    linecolor='white',
    square=False,
    ax=ax,
    yticklabels=True,
    xticklabels=True
)

ax.set_title('Selected Marker Genes Across Cell Types', 
             fontsize=15, fontweight='bold', pad=10)

plt.xticks(rotation=90, ha='center', fontsize=10)
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()

plt.show()


In [ ]:
#SUP FIGURE 1C
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')

# Load single-cell data
adata_sc_new = sc.read_h5ad("/Users/yashkulkarni/cellxgene_data/processed_deconvolution_sc_spleen_240713_fixed_full_data.h5ad")

print(f"Total cells: {adata_sc_new.shape[0]}")
print(f"Timepoints: {adata_sc_new.obs['timepoint'].unique()}")

In [ ]:
#SUP FIGURE 1C MYELOIDS
# Define myeloid cell types
myeloid_types_current = [
    'Myeloid_monocyte', 'Myeloid_Macrophage', 'Myeloid_moDC', 'Myeloid_migratory',
    'Myeloid_DC2', 'Myeloid_Macrophage-CXCL9/10', 'Myeloid_pDC', 'Myeloid_cDC1',
    'Myeloid_marginalzone', 'Myeloid_moDC-CXCL9/10'
]

# Mapping to display names
name_mapping_for_display = {
    'Myeloid_monocyte': 'Monocyte', 'Myeloid_Macrophage': 'Mac-A',
    'Myeloid_moDC': 'moDC-A', 'Myeloid_migratory': 'MigDC', 'Myeloid_DC2': 'cDC2',
    'Myeloid_Macrophage-CXCL9/10': 'Mac-B', 'Myeloid_pDC': 'pDC', 'Myeloid_cDC1': 'cDC1',
    'Myeloid_marginalzone': 'Mac-F480', 'Myeloid_moDC-CXCL9/10': 'moDC-B'
}

# Gene list
gene_order = [
    'Relt',
    'Ccl6',
    'Il1b',
    'Csf3r',
    'S100a8',
    'S100a9',
    'Ccl9',
    'Ccr2',
    'F13a1',
    'Fn1',
    'Chil3',
    'Plcb1',
    'Il27',
    'Ace',
    'Ear2',
    'Cxcl16',
    'H2-M2',
    'Ccl5',
    'Cacnb3',
    'Ccr7',
    'Clec4a4',
    'Rtn1',
    'Ccnd1',
    'Dscam',
    'Tbc1d4',
    'Ifng',
    'Mgl2',
    'Gng2',
    'Irf8',
    'Ccr5',
    'Siglech',
    'Tcf4',
    'Clec9a',
    'Tlr3',
    'Xcr1',
    'Cadm1',
    'Cxcl9',
    'Adgre1',
    'Il18',
    'Mrc1',
    'C1qc',
    'Cd5l',
    'Vcam1',
    'Slamf8',
    'Cxcl10',
    'Mmp14',
    'Cxcr3',
    'Il10',
    'Pdlim1',
    'Il12p40',
    'Il12a',
    'Il12b',
    'Cd86',
    'Cd80',
    
]


# Cell type display order
cell_type_order = ['Monocyte', 'Mac-A', 'moDC-A', 'MigDC', 'cDC2', 
                   'Mac-B', 'pDC', 'cDC1', 'Mac-F480', 'moDC-B']

# Filter for infected and uninfected
adata_infected = adata_sc_new[adata_sc_new.obs['timepoint'] == '3wk'].copy()
adata_uninfected = adata_sc_new[adata_sc_new.obs['timepoint'] == '0wk'].copy()

print(f"Infected cells: {adata_infected.shape[0]}")
print(f"Uninfected cells: {adata_uninfected.shape[0]}")

# Filter for myeloid cells
adata_myeloid_inf = adata_infected[adata_infected.obs['celltypes_redo'].isin(myeloid_types_current)].copy()
adata_myeloid_uninf = adata_uninfected[adata_uninfected.obs['celltypes_redo'].isin(myeloid_types_current)].copy()

print(f"\nMyeloid cells - Infected: {adata_myeloid_inf.shape[0]}")
print(f"Myeloid cells - Uninfected: {adata_myeloid_uninf.shape[0]}")

# Check genes present
genes_present = [g for g in gene_order if g in adata_myeloid_inf.var_names]
print(f"\nGenes present: {len(genes_present)}/{len(gene_order)}")

# Function to calculate mean expression per cell type
def calculate_expression(adata_myeloid, genes_present):
    adata_subset = adata_myeloid[:, genes_present].copy()
    expr_matrix = []
    cell_type_labels = []
    
    for cell_type in myeloid_types_current:
        cells_mask = adata_subset.obs['celltypes_redo'] == cell_type
        cells = adata_subset[cells_mask]
        
        if cells.shape[0] > 0:
            X = cells.X.toarray() if hasattr(cells.X, 'toarray') else cells.X
            mean_expr = X.mean(axis=0)
            expr_matrix.append(mean_expr)
            cell_type_labels.append(name_mapping_for_display[cell_type])
    
    return pd.DataFrame(expr_matrix, index=cell_type_labels, columns=genes_present)

# Calculate expression for both conditions
expr_df_inf = calculate_expression(adata_myeloid_inf, genes_present)
expr_df_uninf = calculate_expression(adata_myeloid_uninf, genes_present)

# Z-score normalize separately
expr_df_inf_norm = expr_df_inf.apply(zscore, axis=0)
expr_df_uninf_norm = expr_df_uninf.apply(zscore, axis=0)

print("\n✅ Expression data calculated and normalized!")

In [ ]:
#SUP FIGURE 1C MYELOIDS
from matplotlib.colors import TwoSlopeNorm

# Calculate expression for COMBINED (all cells, both timepoints)
adata_myeloid_combined = adata_sc_new[adata_sc_new.obs['celltypes_redo'].isin(myeloid_types_current)].copy()

print(f"Myeloid cells - Combined: {adata_myeloid_combined.shape[0]}")

# Calculate mean expression
expr_df_combined = calculate_expression(adata_myeloid_combined, genes_present)

# Z-score normalize
expr_df_combined_norm = expr_df_combined.apply(zscore, axis=0)

# Reindex by cell type order first
expr_ordered_by_celltype = expr_df_combined_norm.reindex(index=cell_type_order)

# ORDER GENES BY PEAK EXPRESSION
# For each gene, find which cell type has the highest z-score
gene_peak_celltype = []
for gene in genes_present:
    peak_celltype_idx = expr_ordered_by_celltype[gene].idxmax()
    # Get the position of this cell type in cell_type_order
    peak_position = cell_type_order.index(peak_celltype_idx)
    # Also store max z-score to break ties
    max_zscore = expr_ordered_by_celltype[gene].max()
    gene_peak_celltype.append((peak_position, -max_zscore, gene))  # negative for descending sort

# Sort genes by peak cell type position (creates diagonal pattern)
gene_peak_celltype.sort()
genes_ordered = [gene for _, _, gene in gene_peak_celltype]

print(f"\nGenes reordered by peak expression:")
print(f"First 5 genes: {genes_ordered[:5]}")
print(f"Last 5 genes: {genes_ordered[-5:]}")

# Create final ordered expression matrix
expr_ordered = expr_ordered_by_celltype[genes_ordered]

# Plot with corrected z-score scaling
fig, ax = plt.subplots(figsize=(14, 5))

# Use TwoSlopeNorm to center at 0
vmin = expr_ordered.min().min()
vmax = expr_ordered.max().max()

sns.heatmap(expr_ordered, cmap='RdBu_r',
            norm=TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax),
            cbar_kws={'label': 'Z-score', 'shrink': 0.7, 'pad': 0.01},
            linewidths=0,
            ax=ax,
            yticklabels=True, 
            xticklabels=True)

ax.set_title('Selected Genes Across Myeloid Cell Types', 
             fontsize=15, fontweight='bold', pad=10)
ax.set_xlabel('Genes', fontsize=12, fontweight='bold')
ax.set_ylabel('Cell Type', fontsize=12, fontweight='bold')

plt.xticks(rotation=90, ha='center', fontsize=9)
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()
plt.savefig('myeloid_gene_expression_COMBINED_diagonal.png', dpi=300, bbox_inches='tight')
plt.savefig('myeloid_gene_expression_COMBINED_diagonal.pdf', bbox_inches='tight')

print("\n📊 Saved: myeloid_gene_expression_COMBINED_diagonal.png/pdf")
print(f"Z-score range: {vmin:.2f} to {vmax:.2f}")
plt.show()

print("\n✅ Diagonal pattern heatmap complete!")

In [ ]:
#FIGURE 4B
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse
import anndata as ad
from collections import OrderedDict

# Load cell2location data
c2l_adata = sc.read_h5ad('/Users/yashkulkarni/cellxgene_data/cell2location_infected_030525_yk.h5ad')

# Load both 10X matrices
print("Loading V1S1 matrix...")
adata_v1s1 = sc.read_10x_h5('/Volumes/YK_Robey/spleen_visium/Spleen_Visium_Exp3A_V1S1_3wk_infected/outs/filtered_feature_bc_matrix.h5')
adata_v1s1.var_names_make_unique()

print("Loading V1S2 matrix...")
adata_v1s2 = sc.read_10x_h5('/Volumes/YK_Robey/spleen_visium/Spleen_Visium_Exp3A_V1S2_3wk_infected/outs/filtered_feature_bc_matrix.h5')
adata_v1s2.var_names_make_unique()

c2l_parsed = [(bc, bc.split('_')[0], '_'.join(bc.split('_')[1:])) for bc in c2l_adata.obs_names]
unique_suffixes = set(suffix for _, _, suffix in c2l_parsed)
print(f"Cell2location sample suffixes: {unique_suffixes}")

v1s1_set = set(adata_v1s1.obs_names)
v1s2_set = set(adata_v1s2.obs_names)

suffix_to_adata = {}
for suffix in sorted(unique_suffixes):
    bases = {base for _, base, s in c2l_parsed if s == suffix}
    overlap_v1s1 = len(bases & v1s1_set)
    overlap_v1s2 = len(bases & v1s2_set)
    if overlap_v1s1 >= overlap_v1s2:
        suffix_to_adata[suffix] = adata_v1s1
        print(f"  '{suffix}' -> V1S1 ({overlap_v1s1} overlapping barcodes)")
    else:
        suffix_to_adata[suffix] = adata_v1s2
        print(f"  '{suffix}' -> V1S2 ({overlap_v1s2} overlapping barcodes)")

matched_c2l = []
for c2l_bc, base_bc, suffix in c2l_parsed:
    sample = suffix_to_adata.get(suffix)
    if sample is not None and base_bc in set(sample.obs_names):
        matched_c2l.append(c2l_bc)

print(f"Found {len(matched_c2l)} matched spots")

c2l_sub = c2l_adata[matched_c2l].copy()

sample_groups = OrderedDict()
for c2l_bc in matched_c2l:
    base_bc = c2l_bc.split('_')[0]
    suffix = '_'.join(c2l_bc.split('_')[1:])
    sample = suffix_to_adata[suffix]
    key = id(sample)
    if key not in sample_groups:
        sample_groups[key] = {'adata': sample, 'base_bcs': [], 'c2l_bcs': []}
    sample_groups[key]['base_bcs'].append(base_bc)
    sample_groups[key]['c2l_bcs'].append(c2l_bc)

parts = []
for group in sample_groups.values():
    part = group['adata'][group['base_bcs']].copy()
    part.obs_names = pd.Index(group['c2l_bcs'])
    parts.append(part)

adata_10x_sub = ad.concat(parts, join='outer', fill_value=0)
adata_10x_sub = adata_10x_sub[matched_c2l].copy()


cluster_col = 'paper_clusters'
adata_10x_sub.obs[cluster_col] = c2l_sub.obs[cluster_col].values
adata_10x_sub.obs[cluster_col] = adata_10x_sub.obs[cluster_col].astype('category')

print(f"10X subset shape: {adata_10x_sub.shape}")

genes_of_interest = [
    'Cxcl9','Cxcl10', 'Cxcl16', 'Cxcr3','Ccl5','Ccl6','Ccr5','Ccr6','Ccr2',
    'Ifng','Ifngr1','Ifngr2',
    'Il18','Il18bp','Il18r1','Il18rap',
    'Ebi3','Il6st','Il27ra','Il27', 'Icos', 'Icosl', 'Jag1', 'Notch2'
]

present_genes = []
for gene in genes_of_interest:
    for var_gene in adata_10x_sub.var_names:
        if gene.lower() == var_gene.lower():
            present_genes.append(var_gene)
            break

print(f"Found genes: {present_genes}")

gene_indices = [list(adata_10x_sub.var_names).index(gene) for gene in present_genes]
X = adata_10x_sub.X[:, gene_indices]
if sparse.issparse(X):
    X = X.toarray()

expr_df = pd.DataFrame(X, columns=present_genes, index=adata_10x_sub.obs_names)
expr_df[cluster_col] = adata_10x_sub.obs[cluster_col].values

cluster_means = expr_df.groupby(cluster_col, observed=True)[present_genes].mean()

Z = cluster_means.T.apply(lambda row: (row - row.mean()) / (row.std() + 1e-9), axis=1).T

# Enforce cluster order: WP A-D then RP A-E
cluster_order = [
    'WP-A: TZ', 'WP-B: BZ', 'WP-C: MZ', 'WP-D: GC',
    'RP-A: GZMK', 'RP-B: H_MK', 'RP-C: NGP', 'RP-D: F480', 'RP-E: Rhag'
]
cluster_order = [c for c in cluster_order if c in Z.index]
Z = Z.loc[cluster_order]

# Sort genes so each gene sits under its highest-expressing cluster,
# producing a diagonal pattern from top-left to bottom-right
gene_peak_cluster = Z.idxmax(axis=0)
cluster_rank = {c: i for i, c in enumerate(cluster_order)}
ordered_genes = sorted(present_genes, key=lambda g: (cluster_rank.get(gene_peak_cluster[g], 999), -Z.loc[gene_peak_cluster[g], g]))
Z = Z[ordered_genes]

print(f"Z-score matrix shape: {Z.shape}")
print(f"Using {len(matched_c2l)} spots from both V1S1 and V1S2")

sns.set_theme(style="white", context="talk")
fig_w = max(14, 0.45 * len(ordered_genes))
fig_h = max(8, 0.55 * len(Z))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.heatmap(
    Z,
    ax=ax,
    cmap='RdBu_r', center=0,
    xticklabels=ordered_genes, yticklabels=Z.index,
    cbar_kws={'label': 'Scaled Expression', 'shrink': 0.6, 'pad': 0.02},
    linewidths=0.1, linecolor='white'
)
ax.grid(False)

ax.set_xlabel('Genes', fontsize=18, fontweight='bold')
ax.set_ylabel('Clusters', fontsize=18, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center', fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=16)

plt.title('Gene Expression in Infected Visium Clusters', fontsize=20, fontweight='bold', pad=20)

for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Cluster marker heatmaps: infected vs uninfected (separate panels) ---
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

# -------------------- user paths --------------------
PATH_INFECTED = "/Users/yashkulkarni/cellxgene_data/cell2location_infected_030525_yk.h5ad"
PATH_UNINFECTED = "/Users/yashkulkarni/cellxgene_data/cell2location_uninfected_030525_yk.h5ad"
CLUSTER_KEY = "paper_clusters"  # column in adata.obs

# -------------------- manual genes + data labels --------------------
# Keys = short labels for brackets/rows; values = genes (only those present are plotted).
manual_gene_lists = OrderedDict(
    [
        ("TZ", ["Ccl21a", "Trbc1", "Thy1"]),
        ("BZ", ["Blk", "Ly6d", "Cd38"]),
        ("2nd Fol", ["Aicda", "Neil1", "Pou2af1"]),
        ("MZ", ["Marco", "Gm2a", "Irf1", "Ltc4s", "Ccl4", "Col23a1", "Ccl24", "Igfbp2", "Vsig10"]),
        ("RP-A-PC", ["Igha", "Jchain", "Iglc2"]),
        ("RP-B-MK", ["Ppbp", "Thbs1", "Pf4"]),
        ("RP-C-Neut", ["S100a9", "Camp", "Ngp"]),
        ("RP-D", ["Ugcg", "Irf9", "Ets2", "Neu1"]),
        ("RP-E-RBC", ["Slc4a1", "Alas2", "Apol11b"]),
    ]
)

# Map short label -> exact cluster string in adata.obs[CLUSTER_KEY]
cluster_name_mapping = {
    "TZ": "WP-A: TZ",
    "BZ": "WP-B: BZ",
    "MZ": "WP-C: MZ",
    "2nd Fol": "WP-D: GC",
    "RP-A-PC": "RP-A: GZMK",
    "RP-B-MK": "RP-B: H_MK",
    "RP-C-Neut": "RP-C: NGP",
    "RP-D": "RP-D: F480",
    "RP-E-RBC": "RP-E: Rhag",
}

# Optional: prettier y-axis labels (e.g. "RP-A - GZMK") — set None to use short keys only
def display_cluster_name(short_key: str, obs_name: str) -> str:
    suffix = obs_name.split(":", 1)[-1].strip()
    prefix = obs_name.split(":", 1)[0].strip()
    return f"{prefix} - {suffix}"


def plot_condition_heatmap(adata, condition_label: str, save_prefix: str | None = None):
    """condition_label: 'infected' or 'uninfected' (used in title/colorbar)."""
    existing_new, existing_old = [], []
    for short_name, old_name in cluster_name_mapping.items():
        if short_name not in manual_gene_lists:
            continue
        if old_name not in adata.obs[CLUSTER_KEY].values:
            continue
        n = int(np.sum(adata.obs[CLUSTER_KEY].astype(str) == old_name))
        if n == 0:
            continue
        existing_new.append(short_name)
        existing_old.append(old_name)

    if not existing_old:
        raise ValueError(f"No clusters found for {condition_label}.")

    condition_genes, gene_to_cluster = [], {}
    for short_name in existing_new:
        for g in manual_gene_lists[short_name]:
            if g in adata.var_names:
                condition_genes.append(g)
                gene_to_cluster[g] = short_name

    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    var_ix = {g: i for i, g in enumerate(adata.var_names)}

    mat = np.zeros((len(existing_old), len(condition_genes)))
    for j, g in enumerate(condition_genes):
        gi = var_ix[g]
        for i, old_name in enumerate(existing_old):
            m = adata.obs[CLUSTER_KEY].astype(str).values == old_name
            mat[i, j] = float(np.mean(X[m, gi])) if np.any(m) else 0.0

    row_labels = [display_cluster_name(sn, cluster_name_mapping[sn]) for sn in existing_new]
    heatmap_df = pd.DataFrame(mat, index=row_labels, columns=condition_genes)

    # Z-score each column (gene) across clusters — within this condition only
    z = heatmap_df.apply(lambda col: (col - col.mean()) / col.std(ddof=0), axis=0).fillna(0.0)

    n_g, n_c = len(condition_genes), len(row_labels)
    fig_w = max(10, n_g * 0.22)
    fig_h = max(6, n_c * 0.45)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    vmin, vmax = float(z.min().min()), float(z.max().max())
    sns.heatmap(
        z,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        cbar_kws={"label": f"Z-score normalized log expression\n({condition_label} only)", "shrink": 0.65},
        xticklabels=True,
        yticklabels=True,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
    )

    ax.set_xticklabels(condition_genes, rotation=90, ha="center", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, ha="right", fontsize=10)

    # vertical separators + top brackets per gene group
    boundaries, labels = [], []
    pos = 0
    for short_name in existing_new:
        grp = [g for g in condition_genes if gene_to_cluster.get(g) == short_name]
        if not grp:
            continue
        start, end = pos, pos + len(grp)
        boundaries.append((start, end))
        labels.append(short_name.split("-")[-1] if "-" in short_name else short_name)  # e.g. GZMK from RP-A-PC -> use short token
        pos = end
        if end < len(condition_genes):
            ax.axvline(x=end, color="white", linewidth=2)

    for i in range(1, len(row_labels)):
        ax.axhline(y=i, color="white", linewidth=1.2)

    trans = ax.get_xaxis_transform()
    for lab, (start, end) in zip(labels, boundaries):
        ax.plot([start + 0.1, end - 0.1], [1.03, 1.03], color="black", lw=1.1, transform=trans, clip_on=False)
        ax.plot([start + 0.1, start + 0.1], [1.01, 1.03], color="black", lw=1.1, transform=trans, clip_on=False)
        ax.plot([end - 0.1, end - 0.1], [1.01, 1.03], color="black", lw=1.1, transform=trans, clip_on=False)
        mid = 0.5 * (start + end)
        ax.text(mid, 1.05, lab, ha="center", va="bottom", rotation=90, fontsize=9, fontweight="bold",
                transform=trans, clip_on=False)

    cond_title = "Infected" if condition_label.lower().startswith("inf") else "Uninfected"
    ax.set_title(f"Cluster Markers: {cond_title} Condition", fontsize=14, fontweight="bold", pad=72)
    ax.set_xlabel("Genes", fontsize=11, fontweight="bold")
    ax.set_ylabel("Clusters", fontsize=11, fontweight="bold")
    plt.tight_layout()

    if save_prefix:
        fig.savefig(f"{save_prefix}.png", dpi=300, bbox_inches="tight", facecolor="white")
        fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight", facecolor="white")
        heatmap_df.to_csv(f"{save_prefix}_expression_matrix.csv")
    return fig, heatmap_df, z


# -------------------- run --------------------
adata_inf = sc.read_h5ad(PATH_INFECTED)
adata_un = sc.read_h5ad(PATH_UNINFECTED)

fig1, df1, z1 = plot_condition_heatmap(adata_inf, "infected", save_prefix="COMPACT_infected_heatmap")
plt.show()

fig2, df2, z2 = plot_condition_heatmap(adata_un, "uninfected", save_prefix="COMPACT_uninfected_heatmap")
plt.show()